In [1]:
# %load_ext autoreload
# %autoreload 2

from constants_and_utils import *

from generate_personas import *
from generate_networks import *
from analyze_networks import *
from plotting import *
from network_datasets import *

import matplotlib.pyplot as plt

# Analyze networks

In [5]:
network_df = pd.read_csv(f'stats/real/network_metrics.csv')
metric_dfs = [network_df]
homophily_dfs = []

# load network metrics and homophily for generated networks
# model = 'gpt-3.5-turbo'
# for method in ['global', 'local', 'sequential']:
#     metric_dfs.append(pd.read_csv(f'stats/{method}_{model}/network_metrics.csv'))
#     homophily_dfs.append(pd.read_csv(f'stats/{method}_{model}/homophily.csv'))
# for extension in ['_ALL_SHUFFLED', '_w_interests', '_only_interests', '_temp06', '_temp10', '_prompt_all', '_w_reason']:
#     metric_dfs.append(pd.read_csv(f'stats/sequential_{model}{extension}/network_metrics.csv'))
#     homophily_dfs.append(pd.read_csv(f'stats/sequential_{model}{extension}/homophily.csv'))
    
# model = 'gpt-4o'
# for extension in ['', '_prompt_all', '_n5', '_n5_prompt_all']:
#     metric_dfs.append(pd.read_csv(f'stats/sequential_{model}{extension}/network_metrics.csv'))
#     homophily_dfs.append(pd.read_csv(f'stats/sequential_{model}{extension}/homophily.csv'))

# model = 'llama3.1-8b'
# for extension in ['', '_n5']:
#     metric_dfs.append(pd.read_csv(f'stats/sequential_{model}{extension}/network_metrics.csv'))
#     homophily_dfs.append(pd.read_csv(f'stats/sequential_{model}{extension}/homophily.csv'))

# model = 'llama3.1-70b'
# for extension in ['', '_n5']:
#     metric_dfs.append(pd.read_csv(f'stats/sequential_{model}{extension}/network_metrics.csv'))
#     homophily_dfs.append(pd.read_csv(f'stats/sequential_{model}{extension}/homophily.csv'))

# model = 'gemma2-9b'
# for extension in ['']:
#     metric_dfs.append(pd.read_csv(f'stats/sequential_{model}{extension}/network_metrics.csv'))
#     homophily_dfs.append(pd.read_csv(f'stats/sequential_{model}{extension}/homophily.csv'))

# model = 'gemma2-27b'
# for extension in ['', '_n5']:
#     metric_dfs.append(pd.read_csv(f'stats/sequential_{model}{extension}/network_metrics.csv'))
#     homophily_dfs.append(pd.read_csv(f'stats/sequential_{model}{extension}/homophily.csv'))

model = 'ollama/gemma4:26b'
for extension in ['_w_reason']:
    metric_dfs.append(pd.read_csv(f'stats/sequential_{model}{extension}/network_metrics.csv'))
    homophily_dfs.append(pd.read_csv(f'stats/sequential_{model}{extension}/homophily.csv'))

network_df = pd.concat(metric_dfs)
print(network_df.groupby('save_name').size())

save_name
real                                     1097
sequential_ollama/gemma4:26b_w_reason     157
dtype: int64


In [6]:
homophily_df = pd.concat(homophily_dfs)
print(homophily_df.groupby('save_name').size())

save_name
sequential_ollama/gemma4:26b_w_reason    10
dtype: int64


In [7]:
def rename_network_metrics(n):
    if n == 'prop_nodes_lcc':
        return 'Prop nodes LCC'
    n = n.split('_')
    n = ' '.join(n)
    return n.capitalize()

for n in network_df['metric_name'].unique():
    print(n, rename_network_metrics(n))
    
network_df['metric_name'] = network_df['metric_name'].apply(rename_network_metrics)
network_df.head()

density Density
avg_clustering_coef Avg clustering coef
prop_nodes_lcc Prop nodes LCC
radius Radius
diameter Diameter
avg_shortest_path Avg shortest path
modularity Modularity
degree_centrality Degree centrality
betweenness_centrality Betweenness centrality
closeness_centrality Closeness centrality


,graph_nr,metric_name,_metric_value,save_name,node
0,0.0,Density,0.144444,real,NaN
1,0.0,Avg clustering coef,0.415549,real,NaN
2,0.0,Prop nodes LCC,0.916667,real,NaN
3,0.0,Radius,0.857999,real,NaN
4,0.0,Diameter,1.429998,real,NaN


In [ ]:
# GPT 3.5
x_to_keep = ['Density', 'Avg clustering coef', 'Prop nodes LCC', 'Avg shortest path', 'Modularity']
network_df[network_df['metric_name'].isin(x_to_keep) & network_df.save_name.str.contains('gpt-3.5')].groupby(['save_name', 'metric_name'])['_metric_value'].agg(['count', 'mean', 'sem']).round(3)

In [ ]:
# non-GPT 3.5 
network_df[network_df['metric_name'].isin(x_to_keep) & ~network_df.save_name.str.contains('gpt-3.5')].groupby(['save_name', 'metric_name'])['_metric_value'].agg(['count', 'mean', 'sem']).round(3)

In [ ]:
# main results - GPT 3.5 metrics, separate plots
to_keep = ['real', 'global_gpt-3.5-turbo', 'local_gpt-3.5-turbo', 'sequential_gpt-3.5-turbo']
kept_df = network_df[network_df.save_name.isin(to_keep)]
plot_metrics_separately(kept_df, plot_type='default', x_to_keep=x_to_keep, 
                            simplify_legend=True, legend_mapper=None)

In [ ]:
# GPT 3.5 - main results on homophily
to_keep = ['global_gpt-3.5-turbo', 'local_gpt-3.5-turbo', 'sequential_gpt-3.5-turbo', 'sequential_gpt-3.5-turbo_ALL_SHUFFLED', 'sequential_gpt-3.5-turbo_w_interests']
kept_df = homophily_df[homophily_df.save_name.isin(to_keep) & (homophily_df['demo'] != 'age')]
make_plot(kept_df, plot_homophily=True, plot_type='default', y_lim=(0.7, 2.2), figsize=(14, 5), dodge=0.65)

In [ ]:
# compare GPT 3.5 vs GPT 4o
to_keep = ['real', 'sequential_gpt-3.5-turbo', 'sequential_gpt-4o', 'sequential_gpt-4o_n5']
kept_df = network_df[network_df.save_name.isin(to_keep)]
legend_mapper = {'real': 'Real',
                 'sequential_gpt-3.5-turbo': 'Seq. (GPT-3.5)',
                 'sequential_gpt-4o': 'Seq. (GPT-4o)',
                 'sequential_gpt-4o_n5': 'Seq.+$\lambda$ (GPT-4o)'}
pastel_palette = sns.color_palette("pastel")
colors = {'real': pastel_palette[0],
          'sequential_gpt-3.5-turbo': pastel_palette[3],
          'sequential_gpt-4o': pastel_palette[4],
          'sequential_gpt-4o_n5': pastel_palette[5]}
make_plot(kept_df, plot_homophily=False, plot_type='default', legend_mapper=legend_mapper, palette=colors,
          x_to_keep=x_to_keep, figsize=(11, 4), legend_pos=(1,1))

to_keep.remove('real')
kept_df = homophily_df[homophily_df.save_name.isin(to_keep) & (homophily_df['demo'] != 'age')]
make_plot(kept_df, plot_homophily=True, plot_type='default', legend_mapper=legend_mapper, palette=colors, 
          dodge=0.55, figsize=(11, 5), legend_pos=(1,1))

In [ ]:
# compare GPT 3.5 vs Llama
to_keep = ['real', 'sequential_gpt-3.5-turbo', 'sequential_llama3.1-8b', 'sequential_llama3.1-8b_n5', 'sequential_llama3.1-70b', 'sequential_llama3.1-70b_n5']
kept_df = network_df[network_df.save_name.isin(to_keep)]
legend_mapper = {'real': 'Real', 
                 'sequential_gpt-3.5-turbo': 'Seq. (GPT-3.5)',
                 'sequential_llama3.1-8b': 'Seq. (Llama3.1 8B)',
                 'sequential_llama3.1-8b_n5': 'Seq.+$\lambda$ (Llama3.1 8B)',
                 'sequential_llama3.1-70b': 'Seq. (Llama3.1 70B)',
                 'sequential_llama3.1-70b_n5': 'Seq.+$\lambda$ (Llama3.1 70B)'}
colors = {'real': pastel_palette[0], 
                 'sequential_gpt-3.5-turbo': pastel_palette[3],
                 'sequential_llama3.1-8b': pastel_palette[4],
                 'sequential_llama3.1-8b_n5': pastel_palette[5],
                 'sequential_llama3.1-70b': pastel_palette[6],
                 'sequential_llama3.1-70b_n5': pastel_palette[7]}

make_plot(kept_df, plot_homophily=False, plot_type='default', legend_mapper=legend_mapper, palette=colors,
          x_to_keep=x_to_keep, figsize=(11, 4), legend_pos=(1,1), dodge=0.67)

to_keep.remove('real')
kept_df = homophily_df[homophily_df.save_name.isin(to_keep) & (homophily_df['demo'] != 'age')]
make_plot(kept_df, plot_homophily=True, plot_type='default', legend_mapper=legend_mapper, palette=colors, 
          dodge=0.65, figsize=(11, 5), legend_pos=(1,1))

In [ ]:
# compare GPT 3.5 vs Gemma
to_keep = ['real', 'sequential_gpt-3.5-turbo', 'sequential_gemma2-9b', 'sequential_gemma2-27b', 'sequential_gemma2-27b_n5']
kept_df = network_df[network_df.save_name.isin(to_keep)]
legend_mapper = {'real': 'Real', 
                 'sequential_gpt-3.5-turbo': 'Seq. (GPT-3.5)',
                 'sequential_gemma2-9b': 'Seq. (Gemma2 9B)',
                 'sequential_gemma2-27b': 'Seq. (Gemma2 27B)',
                 'sequential_gemma2-27b_n5': 'Seq.+$\lambda$ (Gemma2 27B)'}
colors = {'real': pastel_palette[0], 
                 'sequential_gpt-3.5-turbo': pastel_palette[3], 
                 'sequential_gemma2-9b': pastel_palette[4], 
                 'sequential_gemma2-27b': pastel_palette[5], 
                 'sequential_gemma2-27b_n5': pastel_palette[6]}
make_plot(kept_df, plot_homophily=False, plot_type='default', legend_mapper=legend_mapper, palette=colors,
          x_to_keep=x_to_keep, figsize=(11, 4), legend_pos=(1,1), dodge=0.65) # , y_lim=(-0.1, 1.9))

to_keep.remove('real')
kept_df = homophily_df[homophily_df.save_name.isin(to_keep) & (homophily_df['demo'] != 'age')]
make_plot(kept_df, plot_homophily=True, plot_type='default', legend_mapper=legend_mapper, palette=colors,
          figsize=(11, 5), dodge=0.6, legend_pos=(1,1))

In [ ]:
# homophily with interests
to_keep = ['sequential_gpt-3.5-turbo', 'sequential_gpt-3.5-turbo_w_interests', 'sequential_gpt-3.5-turbo_only_interests']
legend_mapper = {'sequential_gpt-3.5-turbo': 'Seq.',
                 'sequential_gpt-3.5-turbo_w_interests': 'Seq. w interests',
                 'sequential_gpt-3.5-turbo_only_interests': 'Seq. only interests'}
colors = {'sequential_gpt-3.5-turbo': pastel_palette[3],
                 'sequential_gpt-3.5-turbo_w_interests': pastel_palette[4],
                 'sequential_gpt-3.5-turbo_only_interests': pastel_palette[5]}
kept_df = homophily_df[homophily_df.save_name.isin(to_keep) & (homophily_df['demo'] != 'age')]
make_plot(kept_df, plot_homophily=True, plot_type='default', legend_mapper=legend_mapper, palette=colors,
          figsize=(11, 5), dodge=0.53)

In [ ]:
# compare across temperatures
to_keep = ['real', 'sequential_gpt-3.5-turbo', 'sequential_gpt-3.5-turbo_temp06', 'sequential_gpt-3.5-turbo_temp10']
legend_mapper = {'real': 'Real', 
                 'sequential_gpt-3.5-turbo': 'Seq. (temp=0.8)',
                 'sequential_gpt-3.5-turbo_temp06': 'Seq. (temp=0.6)',
                 'sequential_gpt-3.5-turbo_temp10': 'Seq. (temp=1.0)'}
colors = {'real': pastel_palette[0],
                 'sequential_gpt-3.5-turbo': pastel_palette[3],
                 'sequential_gpt-3.5-turbo_temp06': pastel_palette[4],
                 'sequential_gpt-3.5-turbo_temp10': pastel_palette[5],}
kept_df = network_df[network_df.save_name.isin(to_keep)]
make_plot(kept_df, plot_homophily=False, plot_type='default', legend_mapper=legend_mapper, palette=colors,
          x_to_keep=x_to_keep, figsize=(11, 4), legend_pos=(1.252,1))

# compare different temperatures
to_keep.remove('real')
kept_df = homophily_df[homophily_df.save_name.isin(to_keep) & (homophily_df['demo'] != 'age')]
make_plot(kept_df, plot_homophily=True, plot_type='default', legend_mapper=legend_mapper, palette=colors,
          figsize=(11, 5), legend_pos=(1,1), dodge=0.53)

In [ ]:
# testing "Pay all to all demographics"
to_keep = ['real', 'sequential_gpt-3.5-turbo', 'sequential_gpt-3.5-turbo_prompt_all', 'sequential_gpt-4o', 'sequential_gpt-4o_prompt_all', 'sequential_gpt-3.5-turbo_w_reason']
kept_df = network_df[network_df.save_name.isin(to_keep)]
legend_mapper = {'real': 'Real', 
                 'sequential_gpt-3.5-turbo': 'Seq. (GPT-3.5)',
                 'sequential_gpt-3.5-turbo_prompt_all': 'Seq.+"all" (GPT-3.5)',
                 'sequential_gpt-4o': 'Seq. (GPT-4o)',
                 'sequential_gpt-4o_prompt_all': 'Seq.+"all" (GPT-4o)',
                 'sequential_gpt-3.5-turbo_w_reason': 'Seq.+"reason" (GPT-3.5)'}
colors = {'real': pastel_palette[0],
                 'sequential_gpt-3.5-turbo': pastel_palette[3],
                 'sequential_gpt-3.5-turbo_prompt_all': pastel_palette[4],
                 'sequential_gpt-4o': pastel_palette[5],
                 'sequential_gpt-4o_prompt_all': pastel_palette[6],
                 'sequential_gpt-3.5-turbo_w_reason': pastel_palette[7]}
make_plot(kept_df, plot_homophily=False, plot_type='default', legend_mapper=legend_mapper, palette=colors,
          x_to_keep=x_to_keep, figsize=(11, 4), dodge=0.65)

to_keep.remove('real')
kept_df = homophily_df[homophily_df.save_name.isin(to_keep) & (homophily_df['demo'] != 'age')]
make_plot(kept_df, plot_homophily=True, plot_type='default', figsize=(11, 5), legend_mapper=legend_mapper, palette=colors,
          dodge=0.62, legend_pos=(1,1))

In [ ]:
to_keep = ['real', 'global_gpt-3.5-turbo', 'local_gpt-3.5-turbo', 'sequential_gpt-3.5-turbo']
kept_df = network_df[network_df.save_name.isin(to_keep)]

palette = get_pallete(kept_df)
fig, axes = plt.subplots(2, 2, figsize=(8, 6))
fig.subplots_adjust(hspace=0.3)
bins = np.arange(0, 1.01, 0.05)
print(bins)

for i, ax in enumerate(axes.flatten()):
    name = to_keep[i]
    degrees = kept_df[(kept_df.save_name == name) & (kept_df.metric_name == 'Degree centrality')]['_metric_value'].values
    print(name, len(degrees), degrees.max())
    ax.hist(degrees, bins=bins, color=palette[name])
    ax.set_yscale('log')
    if i % 2 == 0:
        ax.set_ylabel('Num nodes (log)', fontsize=14)
    if i >= 2:
        ax.set_xlabel('Degree', fontsize=14)
    ax.set_title(get_short_name(name), fontsize=16)
    ax.grid(alpha=0.2)

In [ ]:
real_df = load_real_homophily(same_group=True)
real_df.sort_values('save_name').round(2)

In [ ]:
# age homophily
fn = os.path.join(PATH_TO_TEXT_FILES, 'us_50_gpt4o_w_interests.json')
with open(fn) as f:
    personas = json.load(f)

method = 'sequential'
model = 'gpt-3.5-turbo'
list_of_G = load_list_of_graphs(f'{method}_{model}', 0, 30, directed=False)
plot_expected_vs_observed_age_gaps(list_of_G, personas)

In [ ]:
def report_isolation_index(method, model):
    list_of_G = load_list_of_graphs(f'{method}_{model}', 0, 30, directed=False)
    print(f'{method}_{model}: found {len(list_of_G)} graphs')
    isolation = []
    exposure_c = []
    exposure_l = []
    for G in list_of_G:
        i, c, l = compute_isolation_index(G, personas)
        isolation.append(i)
        exposure_c.append(c)
        exposure_l.append(l)
    print(f'Isolation: {np.mean(isolation):0.3f}, {np.std(isolation)/np.sqrt(len(isolation)):0.3f}')
    print(f'Avg exposure, conservative: {np.mean(exposure_c):0.3f}, {np.std(exposure_c)/np.sqrt(len(exposure_c)):0.3f}')
    print(f'Avg exposure, liberal: {np.mean(exposure_l):0.3f}, {np.std(exposure_l)/np.sqrt(len(exposure_l)):0.3f}')

model = 'gpt-3.5-turbo'
for method in ['global', 'local', 'sequential']:
    print(method)
    report_isolation_index(method, model)
    print()


In [ ]:
def report_polarization(method, model):
    list_of_G = load_list_of_graphs(f'{method}_{model}', 0, 30, directed=False)
    print(f'{method}_{model}: found {len(list_of_G)} graphs')
    pol = []
    for G in list_of_G:
        p = compute_polarization(G, personas)
        pol.append(p)
    print(f'Polarization: mean={np.mean(pol):0.3f}, se={np.std(pol)/np.sqrt(len(pol)):0.3f}')

model = 'gpt-3.5-turbo'
for method in ['global', 'local', 'sequential']:
    print(method)
    report_polarization(method, model)
    print()


# Comparison to classical models

In [ ]:
metrics = ['Density', 'Avg clustering coef', 'Prop nodes LCC', 'Avg shortest path', 'Modularity', 'Degree centrality']
real_mean = network_df[network_df['save_name'] == 'real'].groupby('metric_name')['_metric_value'].agg(['mean', 'sem'])
real_mean

In [ ]:
n = 50
total_edges = n * (n-1) / 2
real_density = real_mean.loc['Density']['mean'] # real networks' mean density
exp_edges = total_edges * real_density
print(exp_edges)

In [ ]:
# erdos-renyi
p = real_density
edges = []
for s in range(30):
    G = nx.erdos_renyi_graph(n, p, seed=s)
    edges.append(len(G.edges()))
    fn = os.path.join(PATH_TO_TEXT_FILES, f'er_{s}.adj')
    nx.write_adjlist(G, fn)
plt.hist(edges, bins=20)
plt.show()

In [ ]:
# barabasi-albert
n = 50
best_m = None 
best_diff = 1e6
for m in range(1, 8):
    G = nx.barabasi_albert_graph(n, m, seed=s)
    num_edges = len(G.edges())
    diff = np.abs(len(G.edges()) - exp_edges)
    if diff < best_diff:
        best_diff = diff 
        best_m = m 
print('Using m =', best_m)

edges = []
for s in range(30):
    G = nx.barabasi_albert_graph(n, best_m, seed=s)
    edges.append(len(G.edges()))
    fn = os.path.join(PATH_TO_TEXT_FILES, f'ba_{s}.adj')
    nx.write_adjlist(G, fn)
print(set(edges))
plt.hist(edges, bins=20)
plt.show()

In [ ]:
# watts-strogatz 
for k in np.arange(4, 16, 2):
    for p in np.arange(0.05, 0.16, 0.05):
        for s in range(5):
            G = nx.watts_strogatz_graph(n, k, p, seed=s)
            assert len(G.edges()) == (n*k)/2

In [ ]:
# fit k (number of neighbors)
best_k = None 
best_diff = 1e6
p = 0.01
for k in np.arange(8, 13, 2):
    G = nx.watts_strogatz_graph(n, k, p, seed=s)
    num_edges = len(G.edges())
    assert num_edges == (n*k)/2
    diff = np.abs(num_edges - exp_edges)
    print(k, diff)
    if diff < best_diff:
        best_diff = diff 
        best_k = k 
print('Using k =', best_k)

In [ ]:
# fit p (rewiring probability)
best_p = None 
best_diff = 1e6
p_options = np.arange(0.01, 0.5, 0.01)
p_clustering = []
for p in p_options:
    clustering = []
    for s in range(30):
        G = nx.watts_strogatz_graph(n, best_k, p)
        clustering.append(nx.average_clustering(G))
    p_clustering.append(np.mean(clustering))
plt.figure(figsize=(6,4))
plt.plot(p_options, p_clustering)
diff = np.abs(p_clustering - real_mean.loc['Avg clustering coef']['mean'])
plt.plot(p_options, diff)
min_val = p_options[np.argmin(diff)]
best_p = min_val.round(3)
print('best p = ', best_p)
ymin, ymax = plt.ylim()
plt.vlines([min_val], ymin, ymax, color='grey')
plt.show()

In [ ]:
for s in range(30):
    G = nx.watts_strogatz_graph(n, best_k, best_p, seed=s)
    fn = os.path.join(PATH_TO_TEXT_FILES, f'ws_{s}.adj')
    nx.write_adjlist(G, fn)

In [ ]:
# summarize network models
min_seed = 0
max_seed = 29
for model in ['er', 'ba', 'ws']: 
    list_of_G, t1, t2 = load_list_of_graphs(model, min_seed, max_seed+1, directed=False, include_ts=True)
    print(f'{model}: found {len(list_of_G)} graphs ({t1} TO {t2})')
    summarize_network_metrics(list_of_G, None, None, model, demos=False)
    print()

In [ ]:
network_df = pd.read_csv(f'stats/real/network_metrics.csv')
metric_dfs = [network_df]
homophily_dfs = []

# load network metrics and homophily for generated networks
model = 'gpt-3.5-turbo'
for method in ['global', 'local', 'sequential']: # , 'iterative']:
    metric_dfs.append(pd.read_csv(f'stats/{method}_{model}/network_metrics.csv'))

for model in ['er', 'ba', 'ws']:
    metric_dfs.append(pd.read_csv(f'stats/{model}/network_metrics.csv'))

network_df = pd.concat(metric_dfs)
print(network_df.groupby('save_name').size())

In [ ]:
results = []
for method in ['er', 'ba', 'ws', 'global', 'local', 'sequential']:
    if len(method) > 2:
        save_name = f'{method}_gpt-3.5-turbo'
    else:
        save_name = method
    for metric in ['density', 'avg_clustering_coef', 'prop_nodes_lcc', 'avg_shortest_path', 'modularity', 'degree_centrality']:
        mean_diff, mean_diff_norm, ks_stat, ks_pval = compare_network_metrics(network_df, metric, save_name)
        results.append({'method': method, 'metric': metric, 'mean_diff': mean_diff, 'mean_diff_norm': mean_diff_norm,
                        'ks_stat': ks_stat, 'ks_pval': ks_pval})
df = pd.DataFrame(results)
df.round(3)